# CalcPlot Example with Deno

This notebook demonstrates how to use the **calcplot** package via Deno for interactive mathematical visualization and simulation.

## Installation

The calcplot package is available on npm and can be used with Deno.

In [ ]:
// Import calcplot using Deno
import { defineIVP, explore, slider, show, simulate, view } from "../src/index";

console.log("CalcPlot imported successfully!");

## Example: Ballistic Trajectory Simulation

Let's create an interactive simulation of projectile motion with air resistance.

In [ ]:
// Define the ballistic motion model
const ballisticModel = defineIVP({
  state: { x: 0, y: 0, vx: 0, vy: 0 },
  params: { g: 9.81, k: 0 },
  derivatives: {
    x: (s) => s.vx,
    y: (s) => s.vy,
    vx: (s, p) => -p.k * s.vx,
    vy: (s, p) => -p.g - p.k * s.vy
  },
  events: {
    groundHit: {
      when: (s) => s.y,
      then: (s, p) => null,
      once: true
    }
  }
});

In [ ]:
// Create an interactive exploration
await explore(
  ballisticModel,
  {
    params: {
      v0: slider(5, 50, 20, 'Initial Speed (m/s)'),
      angle: slider(0, 90, 45, 'Angle (°)'),
      k: slider(0, 0.5, 0.1, 'Air Resistance')
    },

    initial: (p) => {
      const rad = (p.angle * Math.PI) / 180;
      return {
        x: 0,
        y: 0,
        vx: p.v0 * Math.cos(rad),
        vy: p.v0 * Math.sin(rad)
      };
    },

    view: [
      // Trajectory plot
      view()
        .plot((s) => [s.x, s.y], { color: 'blue', label: 'Trajectory' })
        .grid()
        .axis({ xLabel: 'Range (m)', yLabel: 'Height (m)', aspectRatio: 'equal' }),

      // Velocity and height over time
      view()
        .plot((s) => Math.sqrt(s.vx ** 2 + s.vy ** 2), { color: 'red', label: 'Velocity' })
        .plot((s) => s.y, { color: 'green', label: 'Height' })
        .grid()
        .axis({ xLabel: 'Time (s)', yLabel: 'Value' })
    ]
  }
);

In [ ]:
const ballisticWithoutEvent = defineIVP({
  state: { x: 0, y: 0, vx: 0, vy: 0 },
  params: { g: 9.81, k: 0 },
  derivatives: {
    x: (s) => s.vx,
    y: (s) => s.vy,
    vx: (s, p) => -p.k * s.vx,
    vy: (s, p) => -p.g - p.k * s.vy
  },
});

await explore(
  ballisticWithoutEvent,
  {
    params: {
      v0: slider(5, 50, 20, 'Initial Speed (m/s)'),
      angle: slider(0, 90, 45, 'Angle (°)'),
      k: slider(0, 0.5, 0.1, 'Air Resistance')
    },

    initial: (p) => {
      const rad = p.angle * Math.PI / 180;
      return {
        x: 0,
        y: 0,
        vx: p.v0 * Math.cos(rad),
        vy: p.v0 * Math.sin(rad)
      };
    },

    view: [
      // Trajectory plot
      view()
        .plot((s) => [s.x, s.y], { color: 'blue', label: 'Trajectory' })
        .grid()
        .axis({ xLabel: 'Range (m)', yLabel: 'Height (m)', aspectRatio: 'equal' }),

      // Velocity and height over time
      view()
        .plot((s) => Math.sqrt(s.vx ** 2 + s.vy ** 2), { color: 'red', label: 'Velocity' })
        .plot((s) => s.y, { color: 'green', label: 'Height' })
        .grid()
        .axis({ xLabel: 'Time (s)', yLabel: 'Value' })
    ]
  }
);


## Example: Simple Harmonic Oscillator

Let's also create a simulation of a damped harmonic oscillator.

In [ ]:
// Create an interactive exploration
await explore(
  ballisticModel,
  {
    params: {
      v0: slider(5, 50, 20, 'Initial Speed (m/s)'),
      angle: slider(0, 90, 45, 'Angle (°)'),
      k: slider(0, 0.5, 0.1, 'Air Resistance')
    },

    initial: (p) => {
      const rad = (p.angle * Math.PI) / 180;
      return {
        x: 0,
        y: 0,
        vx: p.v0 * Math.cos(rad),
        vy: p.v0 * Math.sin(rad)
      };
    },

    view: [
      // Trajectory plot
      view()
        .plot((s) => [s.x, s.y], { color: 'blue', label: 'Trajectory' })
        .grid()
        .axis({ xLabel: 'Range (m)', yLabel: 'Height (m)', aspectRatio: 'equal' }),

      // Velocity and height over time
      view()
        .plot((s) => Math.sqrt(s.vx ** 2 + s.vy ** 2), { color: 'red', label: 'Velocity' })
        .plot((s) => s.y, { color: 'green', label: 'Height' })
        .grid()
        .axis({ xLabel: 'Time (s)', yLabel: 'Value' })
    ],
  }
);

In [ ]:
// Define harmonic oscillator model
const oscillatorModel = defineIVP({
  state: { x: 1, v: 0 },
  params: { omega: 1, damping: 0.2 },
  derivatives: {
    x: (s) => s.v,
    v: (s, p) => -p.omega * p.omega * s.x - p.damping * s.v
  }
});

// Simulate oscillator trajectory - correct fluent API usage
const trajectory = simulate(oscillatorModel)
  .initial({ x: 1, v: 0 })
  .params({ omega: 1, damping: 0.2 })
  .run({ maxTime: 15, dt: 0.05 });

// Safe access with checks
const timesLength = trajectory.times ? trajectory.times.length : 0;
const statesXLength = trajectory.states && trajectory.states.x ? trajectory.states.x.length : 0;
const statesVLength = trajectory.states && trajectory.states.v ? trajectory.states.v.length : 0;

// Create timeline object for view
const timeline = {
  times: trajectory.times || [],
  states: trajectory.states || { x: [], v: [] }
};

// Create ViewBuilder objects
const positionView = view()
  .plot((s) => s.x, { color: 'blue', label: 'Position' })
  .plot((s) => s.v, { color: 'red', label: 'Velocity' })
  .grid()
  .axis({ xLabel: 'Time (s)', yLabel: 'Value' });

const phaseView = view()
  .plot((s) => [s.x, s.v], { color: 'purple', label: 'Phase Space' })
  .grid()
  .axis({ xLabel: 'Position', yLabel: 'Velocity', aspectRatio: 'equal' });

await show(timeline, [positionView, phaseView], { height: 300 });

In [ ]:
await explore(
  oscillatorModel,
  {
    params: {
      amplitude: slider(0.1, 2, 1, 'Initial Amplitude'),
      omega: slider(0.5, 3, 1, 'Angular Frequency (rad/s)'),
      damping: slider(0, 1, 0.2, 'Damping Coefficient')
    },

    initial: (p) => ({
      x: p.amplitude,
      v: 0
    }),

    view: [
      // Position vs Time
      view()
        .plot((s) => s.x, { color: 'blue', label: 'Position' })
        .plot((s) => s.v, { color: 'red', label: 'Velocity' })
        .grid()
        .axis({ xLabel: 'Time (s)', yLabel: 'Value' }),

      // Phase space plot
      view()
        .plot((s) => [s.x, s.v], { color: 'purple', label: 'Phase Space' })
        .grid()
        .axis({ xLabel: 'Position', yLabel: 'Velocity', aspectRatio: 'equal' })
    ],
  }
);